# I-405 Traffic Flow Forecasting 

## 1.Introduction
---

### 1.1 Problem Statement

Traffic congestion on the I-405 Freeway in Los Angeles area is a significant challenge, impacting travel times, air quality, and overall commuter experience. The dynamic nature of traffic flow is influenced by numerous factors, including time of day, day of the week, special events(holidays, constructions), and weather conditions. Predicting future traffic flow accurately can enable better traffic operation management, early warnings for congestion, and optimized route planning.

This study aims to compare and evaluate the performance of three time series models: Vector Autoregression (VAR), Facebook Prophet, and Long Short-Term Memory (LSTM) neural networks in forecasting traffic flow on the I-405 Freeway, while incorporating weather data as exogenous factors. The goal is to identify the most effective model for accurate traffic prediction, leveraging the combination of historical traffic data and weather information.

The models will be evaluated using two common performance metrics: Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). These metrics will assess their prediction accuracy and model robustness. By comparing these models, the study seeks to contribute to the development of more efficient traffic prediction systems that can assist in alleviating congestion and enhancing urban mobility in the Los Angeles area.

### 1.2 Data

In [1]:
# import
import os
from datetime import datetime
import warnings # To suppress FutureWarning for meteostat package

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from meteostat import Point, Hourly
import holidays

from sklearn.impute import KNNImputer

#### Traffic flow data
----

Traffic flow data is from Caltrans Performance Measurement System [(PeMS)](https://pems.dot.ca.gov/). The traffic data is collected from over 39,000 individual detectors. These sensors span the freeway system across all major metropolitan areas of the State of California. PeMs also provides over ten years of data for historical analysis. It integrates a wide variety of information from Caltrans and other local agency systems.

In the data/traffic-data folder, it contains hourly traffic flow data for the segment of I-405 north bound in Los Angeles County from Jan. 01, 2021 to Dec. 31, 2023. Original data stores in excel files in monthly manner.

#### Weather data
---

Weather data is from [Meteostat](https://dev.meteostat.net/python/) Python package. Cells below showed the process of fetching weather data.

In [3]:
# meteostat still use deprecated 'parse_dates' in pd.read_csv
# suppress the FutureWarning for meteostat package
warnings.simplefilter('ignore', category=FutureWarning)

start = datetime(2021, 1, 1)
end = datetime(2024, 1, 1) # (2023, 12, 31) will only show the 0 o'clock
la = Point(34.01696352677399, -118.41688979588534)
weather = Hourly(la, start, end)

In [4]:
weather = weather.fetch()
weather

,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
time,,,,,,,,,,,
2021-01-01 00:00:00,17.8,0.0,30.0,NaN,NaN,350.0,24.1,NaN,1013.0,NaN,2.0
2021-01-01 01:00:00,16.7,-0.1,32.0,0.0,NaN,340.0,24.1,NaN,1013.1,NaN,2.0
2021-01-01 02:00:00,16.1,-0.6,32.0,0.0,NaN,330.0,25.9,NaN,1013.9,NaN,2.0
2021-01-01 03:00:00,16.1,-1.0,31.0,0.0,NaN,330.0,22.3,NaN,1014.0,NaN,2.0
2021-01-01 04:00:00,15.0,-1.6,32.0,0.0,NaN,320.0,20.5,NaN,1013.9,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31 20:00:00,15.0,10.0,72.0,0.0,NaN,40.0,7.0,NaN,1020.0,NaN,3.0
2023-12-31 21:00:00,15.0,10.0,72.0,0.0,NaN,110.0,7.0,NaN,1019.0,NaN,3.0
2023-12-31 22:00:00,15.0,10.0,72.0,0.0,NaN,100.0,7.0,NaN,1018.0,NaN,3.0


In [5]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26281 entries, 2021-01-01 00:00:00 to 2024-01-01 00:00:00
Freq: h
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   temp    26281 non-null  float64
 1   dwpt    26281 non-null  float64
 2   rhum    26281 non-null  float64
 3   prcp    26280 non-null  float64
 4   snow    0 non-null      float64
 5   wdir    26281 non-null  float64
 6   wspd    26281 non-null  float64
 7   wpgt    0 non-null      float64
 8   pres    26281 non-null  float64
 9   tsun    0 non-null      float64
 10  coco    26101 non-null  float64
dtypes: float64(11)
memory usage: 2.4 MB


In [6]:
weather.isnull().sum()

temp        0
dwpt        0
rhum        0
prcp        1
snow    26281
wdir        0
wspd        0
wpgt    26281
pres        0
tsun    26281
coco      180
dtype: int64

In [7]:
weather.to_csv('../data/weather-21-23.csv')

Meteorological Parameters and Units

| Code | Meaning                 |unit    |
|------|-------------------------|--------|
| TEMP | Air Temperature         | °C     |
| DWPT | Dew Point               | °C     |
| PRCP | Total Precipitation     |  mm    |
| WDIR | Wind (From) Direction   | Degrees|
| WSPD | Average Wind Speed      | km/h   |
| WPGT | Wind Peak Gust          | km/h   |
| RHUM | Relative Humidity       | %      |
| PRES | Sea-Level Air Pressure  | hPa    |
| SNOW | Snow Depth              | mm     |
| TSUN | Total Sunshine Duration | Minutes|
| COCO | Weather Condition Code  | N/A    |


Weather Condition Codes

| Code | Weather Condition   |
|------|---------------------|
| 1    | Clear               |
| 2    | Fair                |
| 3    | Cloudy              |
| 4    | Overcast            |
| 5    | Fog                 |
| 6    | Freezing Fog        |
| 7    | Light Rain          |
| 8    | Rain                |
| 9    | Heavy Rain          |
| 10   | Freezing Rain       |
| 11   | Heavy Freezing Rain |
| 12   | Sleet               |
| 13   | Heavy Sleet         |
| 14   | Light Snowfall      |
| 15   | Snowfall            |
| 16   | Heavy Snowfall      |
| 17   | Rain Shower         |
| 18   | Heavy Rain Shower   |
| 19   | Sleet Shower        |
| 20   | Heavy Sleet Shower  |
| 21   | Snow Shower         |
| 22   | Heavy Snow Shower   |
| 23   | Lightning           |
| 24   | Hail                |
| 25   | Thunderstorm        |
| 26   | Heavy Thunderstorm  |
| 27   | Storm               |


#### Holidays
---

Get holiday data from holiday package.

In [8]:
holiday = holidays.CountryHoliday('US', state = 'CA', years = [2021, 2022, 2023])
# convert hoilday from str to datetime for later use
holiday = pd.to_datetime(list(holiday.keys()))

## 2. Data Cleaning
---

### 2.1 Weather data

Drop these columns which are completely missing.
* snow: It almost never snows in Southern California. 
* wpgt: Wind Peak Gust is for daily weather data.
* tsun: Total Sunshine Duration is for daily weather data.

In [9]:
weather = weather.drop(columns=['snow', 'wpgt', 'tsun'])

In [10]:
weather.tail()

,temp,dwpt,rhum,prcp,wdir,wspd,pres,coco
time,,,,,,,,
2023-12-31 20:00:00,15.0,10.0,72.0,0.0,40.0,7.0,1020.0,3.0
2023-12-31 21:00:00,15.0,10.0,72.0,0.0,110.0,7.0,1019.0,3.0
2023-12-31 22:00:00,15.0,10.0,72.0,0.0,100.0,7.0,1018.0,3.0
2023-12-31 23:00:00,15.0,10.0,72.0,0.0,40.0,9.0,1017.0,3.0
2024-01-01 00:00:00,15.0,9.4,69.0,0.0,103.0,13.0,1018.0,3.0


In [11]:
# drop the last row '2024-1-1'
weather = weather.drop(weather.index[-1])

### 2.2 Traffic flow data

In [12]:
df1 = pd.read_excel('../data/traffic-data/2021-01-flow.xlsx', index_col = 'Hour',parse_dates = True)
# df1.index = df1.index.strftime('%Y-%m-%d %H')
df1.head(2)

,771826-ML,717696-ML,718219-ML,717701-ML,717703-ML,718227-ML,717706-ML,717709-ML,716632-ML,771845-ML,...,717823-ML,767367-ML,717825-ML,717827-ML,771808-ML,771767-ML,772011-ML,772024-ML,# Lane Points,% Observed
Hour,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00,690,846,869,1056,1125,1206,1190,1081,1136,1291,...,1072,1407,1142,1107,2530,3291,675,743,5652,68.135173
2021-01-01 01:00:00,793,880,929,1117,1117,1166,1144,1075,1091,1291,...,1151,1445,1190,1140,2655,3487,733,801,5652,68.046709


In [13]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 744 entries, 2021-01-01 00:00:00 to 2021-01-31 23:00:00
Columns: 107 entries, 771826-ML to % Observed
dtypes: float64(1), int64(106)
memory usage: 627.8 KB


In [14]:
df1.drop(df1.columns[-2:], axis=1, inplace=True)

In [15]:
df1.head(2)

,771826-ML,717696-ML,718219-ML,717701-ML,717703-ML,718227-ML,717706-ML,717709-ML,716632-ML,771845-ML,...,767351-ML,717819-ML,717823-ML,767367-ML,717825-ML,717827-ML,771808-ML,771767-ML,772011-ML,772024-ML
Hour,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00,690,846,869,1056,1125,1206,1190,1081,1136,1291,...,1391,1147,1072,1407,1142,1107,2530,3291,675,743
2021-01-01 01:00:00,793,880,929,1117,1117,1166,1144,1075,1091,1291,...,1537,1248,1151,1445,1190,1140,2655,3487,733,801


In [16]:
df2 = pd.read_excel('../data/traffic-data/2021-02-flow.xlsx', index_col = 'Hour',parse_dates = True)
df2.drop(df2.columns[-2:], axis=1, inplace=True)
df2.head(2)

,771826-ML,717696-ML,718219-ML,717701-ML,717703-ML,718227-ML,717706-ML,717709-ML,716632-ML,771845-ML,...,767351-ML,717819-ML,717823-ML,767367-ML,717825-ML,717827-ML,771808-ML,771767-ML,772011-ML,772024-ML
Hour,,,,,,,,,,,,,,,,,,,,,
2021-02-01 00:00:00,468,585,576,736,787,867,1394,699,761,899,...,1268,878,839,1058,888,853,1902,2294,491,422
2021-02-01 01:00:00,322,389,360,511,658,665,979,467,473,684,...,850,599,580,661,530,562,1438,1796,343,289


In [17]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 672 entries, 2021-02-01 00:00:00 to 2021-02-28 23:00:00
Columns: 105 entries, 771826-ML to 772024-ML
dtypes: int64(105)
memory usage: 556.5 KB


In [18]:
(df1.columns == df2.columns).all()

True

In [19]:
files = os.listdir('../data/traffic-data/')
files.sort()

In [20]:
def dataframe_process(folder):
    """
    Processes and combines multiple traffic data Excel files.
    - Read the file into a DataFrame.
    - Set the 'Hour' column as the index and parsing it as dates.
    - Drop the last two columns(column averages from origin excel) of each DataFrame.
    - Combine the processed DataFrames into a single DataFrame.
    - Remove the '-ML' suffix from column names.
    
    Args:
        folder: A list or other iterable of file names (str) in the folder containing the Excel files to process.
    
    Returns:
        DataFrame: A combined DataFrame from the given file folder.
    """
    df_list = []
    
    for file in folder:
        file_path = f'../data/traffic-data/{file}'
        df = pd.read_excel(file_path, index_col = 'Hour', parse_dates = True)
        df.drop(df.columns[-2:], axis=1, inplace=True)
        df_list.append(df)
    
    # The current data has the same columns
    # 
    # df_1_columns = df_list[0].columns
    # for df in df_list[1:]:
    #     if not (df.columns == df_1_columns).all():
    #         print('Columns in dataframes are not the same')
    #         return None

    df_combined = pd.concat(df_list)
    df_combined.columns = df_combined.columns.str.replace('-ML', '')
    return df_combined

In [21]:
df = dataframe_process(files)
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26206 entries, 2021-01-01 00:00:00 to 2023-12-31 23:00:00
Columns: 105 entries, 771826 to 772024
dtypes: int64(105)
memory usage: 21.2 MB


In [22]:
df.isnull().sum().sum()

0

In [23]:
df.index[1:]

DatetimeIndex(['2021-01-01 01:00:00', '2021-01-01 02:00:00',
               '2021-01-01 03:00:00', '2021-01-01 04:00:00',
               '2021-01-01 05:00:00', '2021-01-01 06:00:00',
               '2021-01-01 07:00:00', '2021-01-01 08:00:00',
               '2021-01-01 09:00:00', '2021-01-01 10:00:00',
               ...
               '2023-12-31 14:00:00', '2023-12-31 15:00:00',
               '2023-12-31 16:00:00', '2023-12-31 17:00:00',
               '2023-12-31 18:00:00', '2023-12-31 19:00:00',
               '2023-12-31 20:00:00', '2023-12-31 21:00:00',
               '2023-12-31 22:00:00', '2023-12-31 23:00:00'],
              dtype='datetime64[ns]', name='Hour', length=26205, freq=None)

In [24]:
df.index[:-1]

DatetimeIndex(['2021-01-01 00:00:00', '2021-01-01 01:00:00',
               '2021-01-01 02:00:00', '2021-01-01 03:00:00',
               '2021-01-01 04:00:00', '2021-01-01 05:00:00',
               '2021-01-01 06:00:00', '2021-01-01 07:00:00',
               '2021-01-01 08:00:00', '2021-01-01 09:00:00',
               ...
               '2023-12-31 13:00:00', '2023-12-31 14:00:00',
               '2023-12-31 15:00:00', '2023-12-31 16:00:00',
               '2023-12-31 17:00:00', '2023-12-31 18:00:00',
               '2023-12-31 19:00:00', '2023-12-31 20:00:00',
               '2023-12-31 21:00:00', '2023-12-31 22:00:00'],
              dtype='datetime64[ns]', name='Hour', length=26205, freq=None)

In [25]:
# Calculate the time difference between timestamps
time_diffs = df.index[1:] - df.index[:-1] # could use shift

print(time_diffs.value_counts())

Hour
0 days 01:00:00    26201
0 days 02:00:00        3
3 days 00:00:00        1
Name: count, dtype: int64


In [26]:
# full range of timestamps
full_index = pd.date_range(start='2021-01-01 00:00:00', end='2023-12-31 23:00:00', freq='h')

missing_timestamps = full_index.difference(df.index)

print('Number of missing timestamps:', len(missing_timestamps))
print('Missing timestamps:', missing_timestamps)

Number of missing timestamps: 74
Missing timestamps: DatetimeIndex(['2021-03-14 02:00:00', '2022-03-13 02:00:00',
               '2023-03-12 02:00:00', '2023-12-19 01:00:00',
               '2023-12-19 02:00:00', '2023-12-19 03:00:00',
               '2023-12-19 04:00:00', '2023-12-19 05:00:00',
               '2023-12-19 06:00:00', '2023-12-19 07:00:00',
               '2023-12-19 08:00:00', '2023-12-19 09:00:00',
               '2023-12-19 10:00:00', '2023-12-19 11:00:00',
               '2023-12-19 12:00:00', '2023-12-19 13:00:00',
               '2023-12-19 14:00:00', '2023-12-19 15:00:00',
               '2023-12-19 16:00:00', '2023-12-19 17:00:00',
               '2023-12-19 18:00:00', '2023-12-19 19:00:00',
               '2023-12-19 20:00:00', '2023-12-19 21:00:00',
               '2023-12-19 22:00:00', '2023-12-19 23:00:00',
               '2023-12-20 00:00:00', '2023-12-20 01:00:00',
               '2023-12-20 02:00:00', '2023-12-20 03:00:00',
               '2023-12-20 04:00

In [27]:
# rename index column to time before merge
df.rename_axis('time', inplace=True)

In [28]:
df_combined = pd.merge(weather, df, how='left', on='time')

In [29]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26280 entries, 2021-01-01 00:00:00 to 2023-12-31 23:00:00
Columns: 113 entries, temp to 772024
dtypes: float64(113)
memory usage: 22.9 MB


In [30]:
# add hoilday columns
holiday_index = pd.to_datetime(holiday.date)
index_dates = pd.to_datetime(df_combined.index.date)
df_combined['isholiday'] = index_dates.isin(holiday_index)

In [57]:
# move the holiday to the first column for later convenience 
last_col = df_combined.columns[-1]
df_combined = df_combined[[last_col] + [col for col in df_combined.columns if col != last_col]]

In [58]:
df_combined.isnull().sum()

isholiday     0
temp          0
dwpt          0
rhum          0
prcp          1
             ..
717827       74
771808       74
771767       74
772011       74
772024       74
Length: 114, dtype: int64

In [59]:
df_combined

,isholiday,temp,dwpt,rhum,prcp,wdir,wspd,pres,coco,771826,...,767351,717819,717823,767367,717825,717827,771808,771767,772011,772024
time,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00,True,17.8,0.0,30.0,NaN,350.0,24.1,1013.0,2.0,690.0,...,1391.0,1147.0,1072.0,1407.0,1142.0,1107.0,2530.0,3291.0,675.0,743.0
2021-01-01 01:00:00,True,16.7,-0.1,32.0,0.0,340.0,24.1,1013.1,2.0,793.0,...,1537.0,1248.0,1151.0,1445.0,1190.0,1140.0,2655.0,3487.0,733.0,801.0
2021-01-01 02:00:00,True,16.1,-0.6,32.0,0.0,330.0,25.9,1013.9,2.0,463.0,...,1164.0,873.0,808.0,1037.0,855.0,823.0,2238.0,2938.0,449.0,541.0
2021-01-01 03:00:00,True,16.1,-1.0,31.0,0.0,330.0,22.3,1014.0,2.0,347.0,...,862.0,585.0,541.0,642.0,544.0,551.0,1813.0,2506.0,327.0,408.0
2021-01-01 04:00:00,True,15.0,-1.6,32.0,0.0,320.0,20.5,1013.9,1.0,388.0,...,753.0,501.0,503.0,561.0,476.0,448.0,1711.0,2557.0,286.0,454.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31 19:00:00,False,15.0,9.4,69.0,0.0,86.0,6.0,1021.0,3.0,5138.0,...,3811.0,3646.0,3561.0,3952.0,5138.0,5025.0,4715.0,5658.0,2289.0,2892.0
2023-12-31 20:00:00,False,15.0,10.0,72.0,0.0,40.0,7.0,1020.0,3.0,5011.0,...,3331.0,3203.0,3011.0,3567.0,5011.0,4586.0,4085.0,5097.0,1814.0,2424.0
2023-12-31 21:00:00,False,15.0,10.0,72.0,0.0,110.0,7.0,1019.0,3.0,4732.0,...,2939.0,2763.0,2546.0,3139.0,4732.0,3677.0,3226.0,4128.0,1460.0,1922.0


In [60]:
# use knn imputer to treat missing values
knn_impute = KNNImputer(weights='distance')
df_combined_knn_imputed = knn_impute.fit_transform(df_combined)
df_combined_knn_imputed = pd.DataFrame(df_combined_knn_imputed, columns=df_combined.columns)

In [61]:
df_combined_knn_imputed.head()

,isholiday,temp,dwpt,rhum,prcp,wdir,wspd,pres,coco,771826,...,767351,717819,717823,767367,717825,717827,771808,771767,772011,772024
0,1.0,17.8,0.0,30.0,0.0,350.0,24.1,1013.0,2.0,690.0,...,1391.0,1147.0,1072.0,1407.0,1142.0,1107.0,2530.0,3291.0,675.0,743.0
1,1.0,16.7,-0.1,32.0,0.0,340.0,24.1,1013.1,2.0,793.0,...,1537.0,1248.0,1151.0,1445.0,1190.0,1140.0,2655.0,3487.0,733.0,801.0
2,1.0,16.1,-0.6,32.0,0.0,330.0,25.9,1013.9,2.0,463.0,...,1164.0,873.0,808.0,1037.0,855.0,823.0,2238.0,2938.0,449.0,541.0
3,1.0,16.1,-1.0,31.0,0.0,330.0,22.3,1014.0,2.0,347.0,...,862.0,585.0,541.0,642.0,544.0,551.0,1813.0,2506.0,327.0,408.0
4,1.0,15.0,-1.6,32.0,0.0,320.0,20.5,1013.9,1.0,388.0,...,753.0,501.0,503.0,561.0,476.0,448.0,1711.0,2557.0,286.0,454.0


In [62]:
# datetime index disappear after imputation 
df_combined_knn_imputed.index = df_combined.index
df_combined_knn_imputed.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26280 entries, 2021-01-01 00:00:00 to 2023-12-31 23:00:00
Columns: 114 entries, isholiday to 772024
dtypes: float64(114)
memory usage: 23.1 MB


In [63]:
df_combined_knn_imputed.isnull().sum()

isholiday    0
temp         0
dwpt         0
rhum         0
prcp         0
            ..
717827       0
771808       0
771767       0
772011       0
772024       0
Length: 114, dtype: int64

In [ ]:
# df_combined_knn_imputed.resample('ME').mean().plot(kind='box',figsize=(20,8));

In [64]:
df_combined_knn_imputed

,isholiday,temp,dwpt,rhum,prcp,wdir,wspd,pres,coco,771826,...,767351,717819,717823,767367,717825,717827,771808,771767,772011,772024
time,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00,1.0,17.8,0.0,30.0,0.0,350.0,24.1,1013.0,2.0,690.0,...,1391.0,1147.0,1072.0,1407.0,1142.0,1107.0,2530.0,3291.0,675.0,743.0
2021-01-01 01:00:00,1.0,16.7,-0.1,32.0,0.0,340.0,24.1,1013.1,2.0,793.0,...,1537.0,1248.0,1151.0,1445.0,1190.0,1140.0,2655.0,3487.0,733.0,801.0
2021-01-01 02:00:00,1.0,16.1,-0.6,32.0,0.0,330.0,25.9,1013.9,2.0,463.0,...,1164.0,873.0,808.0,1037.0,855.0,823.0,2238.0,2938.0,449.0,541.0
2021-01-01 03:00:00,1.0,16.1,-1.0,31.0,0.0,330.0,22.3,1014.0,2.0,347.0,...,862.0,585.0,541.0,642.0,544.0,551.0,1813.0,2506.0,327.0,408.0
2021-01-01 04:00:00,1.0,15.0,-1.6,32.0,0.0,320.0,20.5,1013.9,1.0,388.0,...,753.0,501.0,503.0,561.0,476.0,448.0,1711.0,2557.0,286.0,454.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31 19:00:00,0.0,15.0,9.4,69.0,0.0,86.0,6.0,1021.0,3.0,5138.0,...,3811.0,3646.0,3561.0,3952.0,5138.0,5025.0,4715.0,5658.0,2289.0,2892.0
2023-12-31 20:00:00,0.0,15.0,10.0,72.0,0.0,40.0,7.0,1020.0,3.0,5011.0,...,3331.0,3203.0,3011.0,3567.0,5011.0,4586.0,4085.0,5097.0,1814.0,2424.0
2023-12-31 21:00:00,0.0,15.0,10.0,72.0,0.0,110.0,7.0,1019.0,3.0,4732.0,...,2939.0,2763.0,2546.0,3139.0,4732.0,3677.0,3226.0,4128.0,1460.0,1922.0


In [65]:
# round the Weather Condition code
df_combined_knn_imputed['coco'] = df_combined_knn_imputed['coco'].round()

# convert isholiday to integer
df_combined_knn_imputed['isholiday'] = df_combined_knn_imputed['isholiday'].astype(int)

In [66]:
df_combined_knn_imputed.to_csv('../data/cleared-combine-21-23.csv')

In [54]:
# create station information dataframe

In [41]:
df_temp = pd.DataFrame(df.columns, columns=['id'])

In [42]:
df_temp['id'] = df_temp['id'].astype(int)

In [43]:
stations = pd.read_excel('../data/stations.xlsx')

In [44]:
stations.columns = stations.columns.str.lower().str.replace(' ', '_')

In [45]:
stations = stations[['city', 'id', 'ca_pm', 'abs_pm', 'length']]

In [46]:
df_stations = pd.merge(stations, df_temp, how='right', on='id')
df_stations

,city,id,ca_pm,abs_pm,length
0,Long Beach,771826,.11,24.058,0.303
1,Long Beach,717696,.6,24.548,0.495
2,Long Beach,718219,1.1,25.048,0.565
3,Long Beach,717701,1.73,25.678,0.675
4,Long Beach,717703,2.45,26.398,0.505
...,...,...,...,...,...
100,Los Angeles,717827,44.87,68.642,0.915
101,Los Angeles,771808,45.7,69.472,0.665
102,Los Angeles,771767,46.2,69.972,1.120
103,Los Angeles,772011,47.94,71.712,1.050


In [47]:
df_stations.isnull().sum()

city      10
id         0
ca_pm      0
abs_pm     0
length     0
dtype: int64

In [48]:
# fill unincorporated community with other 
df_stations.fillna('other', inplace=True)

In [56]:
df_stations.to_csv('../data/station_cleaned.csv', index=False)